# 05. Chroma Retrieval Test

Load a persist directory under `chromadb/` and run query tests.

In [3]:
from pathlib import Path
import chromadb

print(f"ChromaDB 버전: {chromadb.__version__}")

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CHROMADB_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'chromadb',
    PROJECT_ROOT / 'data' / 'processed' / 'chroma_db',
    PROJECT_ROOT / 'vectordb' / 'chroma',
]


def looks_like_persist_dir(path: Path) -> bool:
    markers = ['chroma.sqlite3', 'data_level0.bin', 'index_metadata.pickle']
    return path.exists() and any((path / marker).exists() for marker in markers)


candidate_roots = [p for p in CHROMADB_ROOT_CANDIDATES if p.exists()]
assert candidate_roots, f'No ChromaDB root found in: {CHROMADB_ROOT_CANDIDATES}'

candidates = []
for root in candidate_roots:
    if looks_like_persist_dir(root):
        candidates.append(root)

    for child in sorted(root.iterdir()):
        if child.is_dir() and looks_like_persist_dir(child):
            candidates.append(child)

assert candidates, f'No persist directory found under: {candidate_roots}'

PERSIST_DIR = candidates[0]
print('PROJECT_ROOT =', PROJECT_ROOT)
print('PERSIST_DIR  =', PERSIST_DIR)

if len(candidates) > 1:
    print('\nOther candidates:')
    for p in candidates[1:]:
        print('-', p)


ChromaDB 버전: 1.5.7
PROJECT_ROOT = D:\AI\projects\dermatology-rag-chatbot
PERSIST_DIR  = D:\AI\projects\dermatology-rag-chatbot\chromadb

Other candidates:
- D:\AI\projects\dermatology-rag-chatbot\chromadb\59e43fe1-3d6e-4747-941a-3f401d3cdcc0
- D:\AI\projects\dermatology-rag-chatbot\data\processed\chroma_db\TL_내과_통합


In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = 'jhgan/ko-sroberta-multitask'
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

client = chromadb.PersistentClient(path=PERSIST_DIR)
raw_collections = client.list_collections()

if not raw_collections:
    raise RuntimeError(f'No collections found in {PERSIST_DIR}')

collection_names = [c.name if hasattr(c, 'name') else str(c) for c in raw_collections]
print('collections =', collection_names)

COLLECTION_NAME = collection_names[0]
collection = client.get_collection(COLLECTION_NAME)
print('selected collection =', COLLECTION_NAME)
print('embedding model =', EMBED_MODEL_NAME)
print('count =', collection.count())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7537.51it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


collections = ['medical_knowledge']
selected collection = medical_knowledge
embedding model = jhgan/ko-sroberta-multitask
count = 28916


In [16]:
def run_query(query: str, n_results: int = 3):
    query_embedding = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances'],
    )

    docs = results.get('documents', [[]])[0]
    metas = results.get('metadatas', [[]])[0]
    dists = results.get('distances', [[]])[0]

    if not docs:
        print('No results')
        return results

    for i, doc in enumerate(docs, start=1):
        meta = metas[i - 1] if i - 1 < len(metas) else {}
        dist = dists[i - 1] if i - 1 < len(dists) else None
        print(f'[{i}] distance={dist}')
        print('metadata:', meta)
        print('document:', (doc or '')[:300])
        print('-' * 80)

    return results

test_query = [
    "심근경색 진단에서 가장 민감하고 특이적인 혈액 검사는 무엇인가?",
    "인체에서 세포 외액의 부피를 주로 결정하는 주요 전해질은 무엇인가?",
    "하지동맥폐색질환(PAD) 환자에서 1차적으로 사용하는 항혈소판제는 무엇인가?",
]

for i, query in enumerate(test_query):
    print(f"\n질문 {i+1}: {query}")
    run_query(query, n_results=3)


질문 1: 심근경색 진단에서 가장 민감하고 특이적인 혈액 검사는 무엇인가?
[1] distance=0.3234502077102661
metadata: {'c_id': '35_3', 'source': 'TS_국문_학회 가이드라인'}
document: 3. 출혈성 질환의 진단은 문진, 신체검진, 검사로 이루어집니다. 문진에서는 출혈 경향, 출혈 유발 요인, 전신적 출혈 장애 여부, 가족력, 투약 병력, 기저 질환 유무를 확인합니다. 신체검진에서는 표재성 출혈(피부, 구강 점막 출혈)과 심부 출혈(관절강 내 출혈, 근육 내 출혈)을 구분하며, 비장 종대 여부를 확인합니다. 검사로는 혈소판 수, PT, aPTT 등을 반드시 시행하며, CBC, 말초 혈액 도말 소견, 혈액 응고 인자 활성도 검사, FDP, D-dimer, 혈소판 응집 검사 등이 자주 이용됩니다. 4. 출혈성 질환의 감
--------------------------------------------------------------------------------
[2] distance=0.34379667043685913
metadata: {'c_id': '37_3', 'source': 'TS_국문_학회 가이드라인'}
document:  장기 침범의 증상 징후가 있을 때 진단됩니다. 남성에서 흔하며, 우연히 발견되기도 하지만 장기 침범에 의한 증상으로 내원하는 경우도 있습니다. 심장, 피부, 신경, 폐, 간 등의 실질 장기가 흔히 침범되며, 특히 심장 침범은 이환 및 사망의 주요 원인이므로 조기에 심초음파를 포함한 평가와 추적 관찰이 필요합니다. 5. 림프종, 백혈병 및 기타 종양 질환에서도 말초 혈액 호산구 증가가 관찰될 수 있습니다. 흔하지는 않지만 자궁경부암, 폐암, 두경부암, 위암, 대장암 등의 고형암에서도 호산구 증가증이 동반될 수 있습니다. 6. 폐 침
-------------------------------------------------------------------------------